# Validate `evaluation_report_multistage.pdf`

Independently reproduces every number and plot in the PDF report, cell by cell, using the same shared helpers the report itself calls (`multistage_report_lib.py`) -- so this is a genuine check that the report is reading real logs correctly, not a second implementation that could quietly drift from it.

- **Section 1**: raw per-model tables (matches each of the report's 5 table pages exactly)
- **Section 2**: the cross-model summary bar chart
- **Section 3**: pick any (model, strategy, loss-norm) and regenerate its headline plot pair (loss convergence + averaged power spectrum) live
- **Section 4**: the 5 "multistage MSE vs best single-stage LPIPS" appendix plots
- **Section 5**: raw-log spot check -- trace any table cell back to the exact log line it came from

In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import multistage_report_lib as lib
from metrics.kinetic_energy_spectrum import compute_energy_spectrum
from datasets.super_res_dataset import SuperResDataset

print(f"device: {lib.device}")

device: cuda


## Section 1: per-model tables

`gather_model_rows()` is the exact function the PDF's tables are built from -- this should match every row of every table in the report.

In [2]:
all_rows = {model: lib.gather_model_rows(model) for model in lib.MODELS}

tables = {}
for model, rows in all_rows.items():
    df = pd.DataFrame(rows)
    if not df.empty:
        df["strategy_label"] = df["strategy"].map(lib.STRATEGY_LABELS)
        df = df[["strategy_label", "loss_norm", "epsilon", "stage1_final_mse",
                 "multistage_final_mse", "multistage_epochs", "improvement_pct"]]
    tables[model] = df
    print(f"\n=== {model}: {len(rows)}/8 configs ===")
    display(df)


=== SRCNN: 8/8 configs ===


,strategy_label,loss_norm,epsilon,stage1_final_mse,multistage_final_mse,multistage_epochs,improvement_pct
0,MSE only,l1,0.032554,0.001081,0.000664,32,38.575393
1,MSE only,mse,0.032554,0.001081,0.000666,36,38.390379
2,0.6 MSE + 0.4 LPIPS,l1,0.022744,0.000520,0.000485,16,6.730769
3,0.6 MSE + 0.4 LPIPS,mse,0.022744,0.000520,0.000484,16,6.923077
4,0.6 MSE + 0.2 LPIPS + 0.2 KE,l1,0.024306,0.000592,0.000560,16,5.405405
5,0.6 MSE + 0.2 LPIPS + 0.2 KE,mse,0.024306,0.000592,0.000558,16,5.743243
6,0.6 MSE + 0.4 Weighted KE,l1,0.053707,0.002889,0.001146,57,60.332295
7,0.6 MSE + 0.4 Weighted KE,mse,0.053707,0.002889,0.001067,54,63.066805



=== DSCMS: 8/8 configs ===


,strategy_label,loss_norm,epsilon,stage1_final_mse,multistage_final_mse,multistage_epochs,improvement_pct
0,MSE only,l1,0.030791,0.001005,0.000806,25,19.800995
1,MSE only,mse,0.030791,0.001005,0.000700,34,30.348259
2,0.6 MSE + 0.4 LPIPS,l1,0.017320,0.000301,0.000279,16,7.308970
3,0.6 MSE + 0.4 LPIPS,mse,0.017320,0.000301,0.000278,16,7.641196
4,0.6 MSE + 0.2 LPIPS + 0.2 KE,l1,0.021012,0.000486,0.000413,16,15.020576
5,0.6 MSE + 0.2 LPIPS + 0.2 KE,mse,0.021012,0.000486,0.000412,16,15.226337
6,0.6 MSE + 0.4 Weighted KE,l1,0.052515,0.002752,0.001024,104,62.790698
7,0.6 MSE + 0.4 Weighted KE,mse,0.052515,0.002752,0.001613,46,41.388081



=== PRUSR: 8/8 configs ===


,strategy_label,loss_norm,epsilon,stage1_final_mse,multistage_final_mse,multistage_epochs,improvement_pct
0,MSE only,l1,0.027312,0.000769,0.000351,44,54.356307
1,MSE only,mse,0.027312,0.000769,0.000366,40,52.405722
2,0.6 MSE + 0.4 LPIPS,l1,0.018160,0.000332,0.000280,16,15.662651
3,0.6 MSE + 0.4 LPIPS,mse,0.018160,0.000332,0.000276,16,16.867470
4,0.6 MSE + 0.2 LPIPS + 0.2 KE,l1,0.020481,0.000443,0.000372,16,16.027088
5,0.6 MSE + 0.2 LPIPS + 0.2 KE,mse,0.020481,0.000443,0.000368,16,16.930023
6,0.6 MSE + 0.4 Weighted KE,l1,0.038824,0.001427,0.000567,54,60.266293
7,0.6 MSE + 0.4 Weighted KE,mse,0.038824,0.001427,0.000579,48,59.425368



=== SoFoNO: 8/8 configs ===


,strategy_label,loss_norm,epsilon,stage1_final_mse,multistage_final_mse,multistage_epochs,improvement_pct
0,MSE only,l1,0.024452,0.000629,0.000143,28,77.265501
1,MSE only,mse,0.024452,0.000629,0.000125,28,80.127186
2,0.6 MSE + 0.4 LPIPS,l1,0.010370,0.000115,0.000101,16,12.173913
3,0.6 MSE + 0.4 LPIPS,mse,0.010370,0.000115,0.000097,16,15.652174
4,0.6 MSE + 0.2 LPIPS + 0.2 KE,l1,0.011681,0.000136,0.000116,16,14.705882
5,0.6 MSE + 0.2 LPIPS + 0.2 KE,mse,0.011681,0.000136,0.000114,16,16.176471
6,0.6 MSE + 0.4 Weighted KE,l1,0.026754,0.000707,0.000191,37,72.984441
7,0.6 MSE + 0.4 Weighted KE,mse,0.026754,0.000707,0.000167,30,76.379066



=== SRNO: 8/8 configs ===


,strategy_label,loss_norm,epsilon,stage1_final_mse,multistage_final_mse,multistage_epochs,improvement_pct
0,MSE only,l1,0.023746,0.000575,0.000149,32,74.086957
1,MSE only,mse,0.023746,0.000575,0.000172,30,70.086957
2,0.6 MSE + 0.4 LPIPS,l1,0.011730,0.000224,0.000110,16,50.892857
3,0.6 MSE + 0.4 LPIPS,mse,0.011730,0.000224,0.000105,16,53.125000
4,0.6 MSE + 0.2 LPIPS + 0.2 KE,l1,0.011947,0.000145,0.000127,16,12.413793
5,0.6 MSE + 0.2 LPIPS + 0.2 KE,mse,0.011947,0.000145,0.000122,16,15.862069
6,0.6 MSE + 0.4 Weighted KE,l1,0.029529,0.000891,0.000193,39,78.338945
7,0.6 MSE + 0.4 Weighted KE,mse,0.029529,0.000891,0.000208,41,76.655443


## Section 2: cross-model summary bar chart

Same construction as the PDF's page 2: for each (model, strategy), the best improvement of the MSE-trained / L1-trained stage-2 variant.

In [3]:
fig, ax = plt.subplots(figsize=(11, 6.5))
bar_w = 0.2
x = np.arange(len(lib.STRATEGY_ORDER))
colors = plt.cm.tab10(np.linspace(0, 1, len(lib.MODELS)))

for i, model in enumerate(lib.MODELS):
    rows = all_rows[model]
    best_per_strategy = {}
    for r in rows:
        s = r["strategy"]
        if s not in best_per_strategy or r["improvement_pct"] > best_per_strategy[s]:
            best_per_strategy[s] = r["improvement_pct"]
    values = [best_per_strategy.get(s, np.nan) for s in lib.STRATEGY_ORDER]
    ax.bar(x + (i - len(lib.MODELS) / 2 + 0.5) * bar_w, values, width=bar_w, label=model, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels([lib.STRATEGY_LABELS[s] for s in lib.STRATEGY_ORDER], fontsize=9)
ax.set_ylabel("Multistage MSE improvement over Stage 1 alone (%)")
ax.set_title("Multistage improvement by strategy, across all 5 architectures")
ax.axhline(0, color="black", lw=0.8)
ax.grid(True, axis="y", ls="--", alpha=0.4)
ax.legend(fontsize=9, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.show()

## Section 3: regenerate any headline plot pair live

Change `MODEL`/`STRATEGY`/`LOSS_NORM` below and re-run -- rebuilds the exact loss-convergence + averaged-spectrum pair shown in the report's per-model plot page (spectrum uses real forward passes, ~15 samples, light GPU cost).

In [4]:
MODEL = "SRCNN"
STRATEGY = "mse"       # one of: mse, mse_lpips, mse_lpips_ke, mse_wke
LOSS_NORM = "mse"      # mse or l1
N_SPECTRUM_SAMPLES = 15

test_dataset = SuperResDataset(hr_files=lib.TEST_FILES, downsample_factor=1, freq_threshold=10)

Loading datasets...
../hycom_numpy/hycom_2022.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2023.npy: (365, 128, 128, 2)
../hycom_numpy/hycom_2024.npy: (244, 128, 128, 2)
Loaded 974 samples.
Dataset shape: (974, 128, 128, 2)


In [5]:
row = next((r for r in all_rows[MODEL] if r["strategy"] == STRATEGY and r["loss_norm"] == LOSS_NORM), None)
assert row is not None, f"No completed config for {MODEL} {STRATEGY} ({LOSS_NORM})"
print(row)

stage1_log = lib.find_stage1_log(MODEL, STRATEGY)
s1_epochs, s1_val = lib.parse_stage1_log(stage1_log)
offset = s1_epochs[-1] if s1_epochs else 0
ms_log = lib.multistage_log_path(row["weight_path"], MODEL)
ms_epochs, ms_val = lib.parse_multistage_log(ms_log)
ms_epochs_shifted = [offset + e for e in ms_epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"{MODEL} -- {lib.STRATEGY_LABELS[STRATEGY]} (stage 2 {LOSS_NORM.upper()}-trained), "
             f"Final MSE {row['multistage_final_mse']:.4g} ({row['improvement_pct']:+.1f}%)",
             fontsize=12, fontweight="bold")

ax1.semilogy(s1_epochs, s1_val, color="black", lw=1.5, label=f"Stage 1 ({STRATEGY})")
ax1.semilogy(ms_epochs_shifted, ms_val, color="tab:orange", lw=1.5, label=f"Stage 1 + Stage 2 ({LOSS_NORM}-trained)")
if STRATEGY != "mse_lpips":
    lpips_ref_log = lib.find_stage1_log(MODEL, "mse_lpips")
    if lpips_ref_log is not None:
        lp_epochs, lp_val = lib.parse_stage1_log(lpips_ref_log)
        ax1.semilogy(lp_epochs, lp_val, color="tab:red", lw=1.5, label="Single-stage (mse_lpips)")
ax1.axvline(offset, color="gray", ls=":", lw=1, label="Stage 1 -> Stage 2 handoff")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation MSE (log scale)")
ax1.set_title("Loss convergence: staged vs single-stage", fontsize=10)
ax1.grid(True, which="both", ls="--", alpha=0.3)
ax1.legend(fontsize=8)

multistage, stage1, stage2, epsilon = lib.load_multistage(MODEL, row["weight_path"])
n_samples = min(N_SPECTRUM_SAMPLES, len(test_dataset))
indices = np.linspace(0, len(test_dataset) - 1, n_samples, dtype=int)

hr_spectra, s1_spectra, final_spectra = [], [], []
k_bins = None
with torch.no_grad():
    for idx in indices:
        lr, hr = test_dataset[int(idx)]
        lr, hr = lr.unsqueeze(0).to(lib.device), hr.unsqueeze(0).to(lib.device)
        stage1_pred = stage1(lr)
        final_pred = stage1_pred + epsilon * stage2(lr)
        k_bins, e_hr = compute_energy_spectrum(hr.squeeze(0).cpu().numpy())
        _, e_s1 = compute_energy_spectrum(stage1_pred.squeeze(0).cpu().numpy())
        _, e_final = compute_energy_spectrum(final_pred.squeeze(0).cpu().numpy())
        hr_spectra.append(e_hr)
        s1_spectra.append(e_s1)
        final_spectra.append(e_final)

hr_arr, s1_arr, final_arr = np.stack(hr_spectra), np.stack(s1_spectra), np.stack(final_spectra)
hr_mean, hr_std = hr_arr.mean(axis=0), hr_arr.std(axis=0)

ax2.fill_between(k_bins, np.clip(hr_mean - hr_std, 1e-30, None), hr_mean + hr_std,
                  color="gray", alpha=0.4, label="Target (HR, mean +/- std)")
ax2.loglog(k_bins, hr_mean, color="black", lw=1, alpha=0.6)
ax2.loglog(k_bins, s1_arr.mean(axis=0), color="tab:purple", lw=2, ls="--", label="Stage 1 Alone")
ax2.loglog(k_bins, final_arr.mean(axis=0), color="tab:orange", lw=2, label="Stage 1 + Stage 2 (Final)")
ax2.set_xlabel("Wavenumber $k$")
ax2.set_ylabel("$E(k)$")
ax2.set_title(f"Power spectrum, averaged over {n_samples} samples", fontsize=10)
ax2.grid(True, which="both", ls="--", alpha=0.3)
ax2.legend(fontsize=8)
plt.show()

del multistage, stage1, stage2
if lib.device == "cuda":
    torch.cuda.empty_cache()

{'strategy': 'mse', 'loss_norm': 'mse', 'stage1_final_mse': 0.001081, 'stage1_epochs': 51, 'multistage_final_mse': 0.000666, 'multistage_epochs': 36, 'epsilon': 0.03255418864730462, 'weight_path': '/home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_multistage/08_09_2026_152508/output/weights/SRCNN_best.pth', 'improvement_pct': 38.39037927844588}


## Section 4: appendix plots -- multistage MSE vs best single-stage LPIPS, per model

In [6]:
def plot_mse_vs_best_lpips(model):
    stage1_mse_log = lib.find_stage1_log(model, "mse")
    mse_row = next((r for r in all_rows[model] if r["strategy"] == "mse" and r["loss_norm"] == "mse"), None)
    if stage1_mse_log is None or mse_row is None:
        print(f"{model}: mse multistage not finished yet")
        return

    s1_epochs, s1_val = lib.parse_stage1_log(stage1_mse_log)
    offset = s1_epochs[-1] if s1_epochs else 0
    ms_log = lib.multistage_log_path(mse_row["weight_path"], model)
    ms_epochs, ms_val = lib.parse_multistage_log(ms_log)
    ms_epochs_shifted = [offset + e for e in ms_epochs]

    lpips_candidates = []
    for lpips_strategy in ["mse_lpips", "mse_lpips_ke"]:
        log = lib.find_stage1_log(model, lpips_strategy)
        if log is not None:
            epochs, val = lib.parse_stage1_log(log)
            if val:
                lpips_candidates.append((lpips_strategy, epochs, val))
    if not lpips_candidates:
        print(f"{model}: no lpips-family reference available")
        return
    best_lpips_strategy, lp_epochs, lp_val = min(lpips_candidates, key=lambda t: t[2][-1])

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.semilogy(s1_epochs, s1_val, color="black", lw=1.5, label="Stage 1 (mse)")
    ax.semilogy(ms_epochs_shifted, ms_val, color="tab:orange", lw=1.5, label="Stage 1 + Stage 2 (mse, mse-trained)")
    ax.semilogy(lp_epochs, lp_val, color="tab:red", lw=1.5, label=f"Single-stage ({best_lpips_strategy})")
    ax.axvline(offset, color="gray", ls=":", lw=1, label="Stage 1 -> Stage 2 handoff")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation MSE (log scale)")
    ax.set_title(f"{model} -- Loss convergence: multistage MSE vs best single-stage LPIPS")
    ax.grid(True, which="both", ls="--", alpha=0.3)
    ax.legend(fontsize=9)
    plt.show()


for model in lib.MODELS:
    plot_mse_vs_best_lpips(model)

## Section 5: raw-log spot check

Trace any table cell back to the exact log file and line it came from -- for when a number looks surprising and you want to see the source, not just the parsed value.

In [7]:
CHECK_MODEL = "SRCNN"
CHECK_STRATEGY = "mse"
CHECK_LOSS_NORM = "mse"

row = next((r for r in all_rows[CHECK_MODEL]
            if r["strategy"] == CHECK_STRATEGY and r["loss_norm"] == CHECK_LOSS_NORM), None)
assert row is not None

stage1_log = lib.find_stage1_log(CHECK_MODEL, CHECK_STRATEGY)
ms_log = lib.multistage_log_path(row["weight_path"], CHECK_MODEL)

print(f"Stage-1 log: {stage1_log}")
print(f"Multistage log: {ms_log}")
print(f"Checkpoint: {row['weight_path']}")
print()
print("--- last 6 lines of stage-1 log ---")
print("".join(open(stage1_log).readlines()[-6:]))
print("--- last 6 lines of multistage log ---")
print("".join(open(ms_log).readlines()[-6:]))
print()
print(f"Parsed: stage1_final_mse={row['stage1_final_mse']:.6g}, "
      f"multistage_final_mse={row['multistage_final_mse']:.6g}, "
      f"improvement={row['improvement_pct']:+.1f}%")

Stage-1 log: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_05_08_2026/output/logs/SRCNN_loss.txt
Multistage log: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_multistage/08_09_2026_152508/output/logs/SRCNN_multistage_loss.txt
Checkpoint: /home/lcadame/Vinicius_Thesis/Vinicius/models/SRCNN/test_composite_loss_mse_multistage/08_09_2026_152508/output/weights/SRCNN_best.pth

--- last 6 lines of stage-1 log ---
  Val   -> Total: 0.001077 | MSE: 0.001077 | LPIPS: 0.000000 | Kin: 0.000000
----------------------------------------------------------------------
[Duration: 6.1s | Total Duration: 0.09h] Epoch [51/2500] | LR: 0.000100
  Train -> Total: 0.001060 | MSE: 0.001060 | LPIPS: 0.000000 | Kin: 0.000000
  Val   -> Total: 0.001081 | MSE: 0.001081 | LPIPS: 0.000000 | Kin: 0.000000
----------------------------------------------------------------------

--- last 6 lines of multistage log ---
  Val   -> Residual loss: 0.633455 | Final MS